<a href="https://colab.research.google.com/github/Adithya-2010/AuraBuds/blob/main/AuraBuds_Technical_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
"""AuraBuds CHB-MIT proof of concept (offline scalp-EEG analysis).

Install: pip install mne numpy scipy pandas matplotlib
Keep the EDF and matching CHB-MIT summary file beside this script.
This is an experimental anomaly flagger, not a clinical seizure detector.
"""
!pip install mne -q
from pathlib import Path
import re

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from scipy.signal import welch

# ----------------------------- SETTINGS ---------------------------------
EDF_FILE = Path("chb02_05.edf")
SUMMARY_FILE = Path("chb02-summary.txt")
OUTPUT_DIR = Path("aurabuds_output")
CHANNEL = None                       # Example: "T7-P7"; None = auto-select
TEMPORAL_CHANNELS = ["T7-P7", "T8-P8", "T3-T5", "T4-T6"]
WINDOW_SEC = 5                       # Window length
STEP_SEC = 1                         # Time between windows
BASELINE_SEC = 120                   # Seizure-free calibration duration
SEIZURE_BUFFER_SEC = 30              # Keep calibration away from seizures
THRESHOLD_MULTIPLIER = 4.5


def read_seizures():
    """Return (start, end) seizure times for this EDF, in seconds."""
    text = SUMMARY_FILE.read_text(encoding="utf-8", errors="ignore")
    # Split by blocks starting with File Name
    sections = re.split(r"(?im)(?=File Name\s*:)", text)

    target_name = EDF_FILE.name.lower()
    for section in sections:
        name_match = re.search(r"(?im)File Name\s*:\s*([^\r\n]+)", section)
        if not name_match:
            continue
        # Extract just the filename without any folder prefix
        extracted_name = Path(name_match.group(1).strip()).name.lower()
        if extracted_name != target_name:
            continue

        # Check how many seizures are in this file
        num_match = re.search(r"(?im)Number of Seizures in File\s*:\s*(\d+)", section)
        if num_match and int(num_match.group(1)) == 0:
            return []

        starts = re.findall(r"(?im)Seizure(?:\s+\d+)?\s+Start Time\s*:\s*([\d.]+)", section)
        ends = re.findall(r"(?im)Seizure(?:\s+\d+)?\s+End Time\s*:\s*([\d.]+)", section)

        seizures = [(float(a), float(b)) for a, b in zip(starts, ends)]
        return sorted(seizures)

    print(f"Warning: Could not find {EDF_FILE.name} in summary. Assuming 0 seizures.")
    return []


def choose_channel(names):
    if CHANNEL:
        if CHANNEL not in names:
            raise ValueError(f"Channel {CHANNEL!r} not found. Available: {names}")
        return CHANNEL
    available = {name.lower(): name for name in names}
    for candidate in TEMPORAL_CHANNELS:
        if candidate.lower() in available:
            return available[candidate.lower()]
    raise ValueError("No preferred temporal channel found. Set CHANNEL manually.")


def find_baseline(seizures, recording_sec):
    """Find the earliest full baseline interval outside seizure buffer zones."""
    start = 0.0
    while start + BASELINE_SEC <= recording_sec:
        conflict = next(
            ((a, b) for a, b in seizures
             if start < b + SEIZURE_BUFFER_SEC
             and start + BASELINE_SEC > a - SEIZURE_BUFFER_SEC),
            None,
        )
        if conflict is None:
            return start
        start = conflict[1] + SEIZURE_BUFFER_SEC
    raise ValueError("No seizure-free 120-second baseline was found in this recording.")


def features_for_window(signal_uv, fs):
    freqs, psd = welch(signal_uv, fs=fs, nperseg=len(signal_uv))
    slow = psd[(freqs >= 1) & (freqs < 8)].sum()
    total = psd[(freqs >= 1) & (freqs < 30)].sum()
    slow_ratio = slow / total if total > 0 else np.nan
    line_length = np.mean(np.abs(np.diff(signal_uv))) * fs
    return float(slow_ratio), float(line_length)


def make_windows(signal_uv, fs):
    size, step = int(WINDOW_SEC * fs), int(STEP_SEC * fs)
    if len(signal_uv) < size:
        raise ValueError("The recording is shorter than one analysis window.")
    rows = []
    for first in range(0, len(signal_uv) - size + 1, step):
        slow, line = features_for_window(signal_uv[first:first + size], fs)
        rows.append((first / fs, (first + size) / fs, slow, line))
    return pd.DataFrame(rows, columns=["start_s", "end_s", "slow_ratio", "line_length"])


def upper_threshold(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) < 20:
        raise ValueError("Not enough valid windows in the baseline.")
    median = np.median(values)
    mad = 1.4826 * np.median(np.abs(values - median))
    spread = mad if mad > 1e-9 else np.std(values)
    spread = max(spread, abs(median) * 1e-6, 1e-9)
    return median + THRESHOLD_MULTIPLIER * spread


def save_plot(df, seizures, channel):
    time_min = (df["start_s"] + WINDOW_SEC / 2) / 60
    fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
    axes[0].plot(time_min, df["slow_ratio"], label="Slow-wave ratio")
    axes[0].axhline(df["slow_limit"].iloc[0], linestyle="--", label="Threshold")
    axes[0].set_ylabel("1–8 Hz / 1–30 Hz")
    axes[1].plot(time_min, df["line_length"], label="Line length")
    axes[1].axhline(df["line_limit"].iloc[0], linestyle="--", label="Threshold")
    axes[1].set_ylabel("µV/second")
    axes[1].set_xlabel("Recording time (minutes)")
    for axis in axes:
        for start, end in seizures:
            axis.axvspan(start / 60, end / 60, color="orange", alpha=0.25)
        axis.grid(alpha=0.25)
        axis.legend()
    fig.suptitle(f"AuraBuds CHB-MIT PoC | {channel}")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "eeg_analysis.png", dpi=150)
    plt.close(fig)


def main():
    if not EDF_FILE.exists() or not SUMMARY_FILE.exists():
        raise FileNotFoundError("Put the EDF and matching summary file beside this script.")

    seizures = read_seizures()
    raw = mne.io.read_raw_edf(EDF_FILE, preload=True, verbose="ERROR")
    fs = float(raw.info["sfreq"])
    channel = choose_channel(raw.ch_names)
    high_cutoff = min(40.0, fs / 2 - 1)
    if high_cutoff <= 1:
        raise ValueError("Sampling rate is too low for the EEG filter.")

    eeg = raw.copy().pick([channel])
    eeg.filter(1.0, high_cutoff, verbose="ERROR")
    signal_uv = eeg.get_data()[0] * 1e6
    recording_sec = len(signal_uv) / fs

    baseline_start = find_baseline(seizures, recording_sec)
    baseline_end = baseline_start + BASELINE_SEC
    print(f"Channel: {channel} | Sampling rate: {fs:g} Hz")
    print(f"Seizures in file: {len(seizures)} | Baseline: {baseline_start:.0f}–{baseline_end:.0f} s")

    df = make_windows(signal_uv, fs)
    baseline = df[(df.start_s >= baseline_start) & (df.end_s <= baseline_end)]
    slow_limit = min(0.98, upper_threshold(baseline.slow_ratio))
    line_limit = upper_threshold(baseline.line_length)

    # 1. Flag whenever line length spikes significantly
    # (You can also allow a massive slow-wave spike, but line length is key here)
    df["raw_alert"] = (df.line_length > line_limit)

    # 2. Require persistence of 2 or 3 consecutive windows (2-3 seconds)
    # This ignores 1-second transient pops but catches the multi-second seizure
    df["alert"] = df.raw_alert.rolling(3, min_periods=3).sum().eq(3)
    # The alert is available when each window ends, so label that decision time.
    df["seizure_window"] = [
        any(start < row.end_s <= end for start, end in seizures)
        for row in df.itertuples()
    ]
    df["slow_limit"] = slow_limit
    df["line_limit"] = line_limit

    # Do not score the calibration interval or data recorded before it.
    evaluated = df[df.start_s >= baseline_end].copy()
    tested_seizures = [(a, b) for a, b in seizures if a >= baseline_end]
    detected = sum(
        bool((evaluated.alert & (evaluated.end_s > start)
              & (evaluated.end_s <= end)).any())
        for start, end in tested_seizures
    )
    # Count only the beginning of each alert episode, not every alert window.
    alert_starts = df.alert & ~df.alert.shift(fill_value=False)
    false_alarms = int((alert_starts & (df.start_s >= baseline_end)
                        & ~df.seizure_window).sum())
    seizure_seconds = sum(max(0, min(b, recording_sec) - max(a, baseline_end))
                          for a, b in seizures)
    monitored_hours = max(0, recording_sec - baseline_end - seizure_seconds) / 3600
    false_per_hour = false_alarms / monitored_hours if monitored_hours else 0.0

    OUTPUT_DIR.mkdir(exist_ok=True)
    df.to_csv(OUTPUT_DIR / "window_results.csv", index=False)
    save_plot(df, seizures, channel)

    print("\n--- Results (after baseline) ---")
    if tested_seizures:
        print(f"Seizures detected: {detected}/{len(tested_seizures)} "
              f"({100 * detected / len(tested_seizures):.1f}%)")
    else:
        print("No annotated seizures remain after the baseline; detection rate unavailable.")
    print(f"False alarm events per hour: {false_per_hour:.2f}")
    print(f"Saved: {OUTPUT_DIR / 'window_results.csv'} and {OUTPUT_DIR / 'eeg_analysis.png'}")
    print("Experimental scalp-EEG proof of concept only; not a clinical detector.")


if __name__ == "__main__":
    main()


Channel: T7-P7 | Sampling rate: 256 Hz
Seizures in file: 0 | Baseline: 0–120 s

--- Results (after baseline) ---
No annotated seizures remain after the baseline; detection rate unavailable.
False alarm events per hour: 0.00
Saved: aurabuds_output/window_results.csv and aurabuds_output/eeg_analysis.png
Experimental scalp-EEG proof of concept only; not a clinical detector.


In [17]:
"""
AuraBuds CHB-MIT Proof of Concept (Offline Scalp-EEG Analysis)
NYAS Junior Academy - Brain Tech Challenge Track

Dependencies: pip install mne numpy scipy pandas matplotlib
Place the EDF and corresponding CHB-MIT summary text file in the same directory.
"""

from pathlib import Path
import re
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from scipy.signal import welch

# ----------------------------- SETTINGS ---------------------------------
EDF_FILE = Path("chb01_03.edf")
SUMMARY_FILE = Path("chb01-summary.txt")
OUTPUT_DIR = Path("aurabuds_output")
OUTPUT_DIR.mkdir(exist_ok=True)

CHANNEL = None                       # Specific channel name (e.g., "T7-P7"); None = auto-select
TEMPORAL_CHANNELS = ["T7-P7", "T8-P8", "T8-P8-0", "T3-T5", "T4-T6", "F7-T7", "F8-T8"]
WINDOW_SEC = 5                       # Sliding window duration
STEP_SEC = 1                         # Window step size
BASELINE_SEC = 120                   # Seizure-free calibration duration
SEIZURE_BUFFER_SEC = 30              # Buffer distance around known seizures
THRESHOLD_MULTIPLIER = 4.0           # Median + 4.0 * MAD
PERSISTENCE_COUNT = 3                # 3 overlapping windows span 7 seconds of data


def read_seizures():
    """Extract (start, end) timestamps. Raises errors on missing/mismatched annotations."""
    if not SUMMARY_FILE.exists():
        raise FileNotFoundError(f"Summary file not found: {SUMMARY_FILE}")

    text = SUMMARY_FILE.read_text(encoding="utf-8", errors="ignore")
    sections = re.split(r"(?im)(?=File Name\s*:)", text)
    target_name = EDF_FILE.name.lower()

    for section in sections:
        name_match = re.search(r"(?im)File Name\s*:\s*([^\r\n]+)", section)
        if not name_match:
            continue
        if Path(name_match.group(1).strip()).name.lower() != target_name:
            continue

        num_match = re.search(r"(?im)Number of Seizures in File\s*:\s*(\d+)", section)
        annotated_count = int(num_match.group(1)) if num_match else 0
        if annotated_count == 0:
            return []

        starts = [float(s) for s in re.findall(r"(?im)Seizure(?:\s+\d+)?\s+Start Time\s*:\s*([\d.]+)", section)]
        ends = [float(e) for e in re.findall(r"(?im)Seizure(?:\s+\d+)?\s+End Time\s*:\s*([\d.]+)", section)]

        if len(starts) != len(ends) or len(starts) != annotated_count:
            raise ValueError(f"Annotation mismatch in {SUMMARY_FILE.name}: expected {annotated_count} seizures.")

        intervals = sorted(zip(starts, ends))
        if any(b <= a for a, b in intervals):
            raise ValueError(f"Invalid interval detected in {SUMMARY_FILE.name}.")
        return intervals

    # Priority 2: Make missing filename a fatal error rather than silent default
    raise ValueError(f"File {EDF_FILE.name} not found in summary {SUMMARY_FILE.name}.")


def choose_channel(names):
    """Finds best matching temporal channel."""
    if CHANNEL:
        if CHANNEL not in names:
            raise ValueError(f"Channel {CHANNEL!r} not in recording. Available: {names}")
        return CHANNEL
    available = {n.lower(): n for n in names}
    for candidate in TEMPORAL_CHANNELS:
        if candidate.lower() in available:
            return available[candidate.lower()]
    raise ValueError("No preferred temporal channel found. Specify CHANNEL manually.")


def find_baseline(seizures, recording_sec):
    """Identifies the earliest seizure-free 120-second interval."""
    start = 0.0
    while start + BASELINE_SEC <= recording_sec:
        conflict = next(
            ((a, b) for a, b in seizures
             if start < b + SEIZURE_BUFFER_SEC and start + BASELINE_SEC > a - SEIZURE_BUFFER_SEC),
            None
        )
        if conflict is None:
            return start
        start = conflict[1] + SEIZURE_BUFFER_SEC
    raise ValueError("No seizure-free baseline window found.")


def features_for_window(signal_uv, fs):
    """Computes spectral power ratio and signal line length."""
    freqs, psd = welch(signal_uv, fs=fs, nperseg=len(signal_uv))
    slow = psd[(freqs >= 1) & (freqs < 8)].sum()
    total = psd[(freqs >= 1) & (freqs < 30)].sum()
    slow_ratio = float(slow / total) if total > 0 else 0.0
    line_length = float(np.mean(np.abs(np.diff(signal_uv))) * fs)
    return slow_ratio, line_length


def make_windows(signal_uv, fs):
    size, step = int(WINDOW_SEC * fs), int(STEP_SEC * fs)
    rows = []
    for first in range(0, len(signal_uv) - size + 1, step):
        slow, line = features_for_window(signal_uv[first:first + size], fs)
        rows.append((first / fs, (first + size) / fs, slow, line))
    return pd.DataFrame(rows, columns=["start_s", "end_s", "slow_ratio", "line_length"])


def upper_threshold(values):
    """Calculates non-parametric upper limit using Median Absolute Deviation."""
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    med = np.median(v)
    mad = 1.4826 * np.median(np.abs(v - med))
    spread = max(mad if mad > 1e-9 else np.std(v), 1e-6)
    return float(med + THRESHOLD_MULTIPLIER * spread)


def evaluate_rule(df, alert_series, seizures, baseline_end, recording_sec):
    """Evaluates event sensitivity and false alarm rate for a specific alert rule."""
    tested_seizures = [(a, b) for a, b in seizures if a >= baseline_end]
    evaluated = df[df.start_s >= baseline_end]

    detected = 0
    for a, b in tested_seizures:
        if alert_series[(df.end_s > a) & (df.end_s <= b) & (df.start_s >= baseline_end)].any():
            detected += 1

    alert_starts = alert_series & ~alert_series.shift(fill_value=False)
    false_alarms = int((alert_starts & (df.start_s >= baseline_end) & ~df.seizure_window).sum())

    seizure_seconds = sum(max(0, min(b, recording_sec) - max(a, baseline_end)) for a, b in seizures)
    monitored_hours = max(0, recording_sec - baseline_end - seizure_seconds) / 3600.0
    far = false_alarms / monitored_hours if monitored_hours > 0 else 0.0

    return detected, len(tested_seizures), far


def main():
    if not EDF_FILE.exists() or not SUMMARY_FILE.exists():
        raise FileNotFoundError("Missing EDF or summary file.")

    seizures = read_seizures()
    raw = mne.io.read_raw_edf(EDF_FILE, preload=True, verbose="ERROR")
    fs = float(raw.info["sfreq"])
    channel = choose_channel(raw.ch_names)

    high_cut = min(40.0, fs / 2 - 1)
    raw.pick([channel]).filter(1.0, high_cut, verbose="ERROR")
    signal_uv = raw.get_data()[0] * 1e6
    recording_sec = len(signal_uv) / fs

    baseline_start = find_baseline(seizures, recording_sec)
    baseline_end = baseline_start + BASELINE_SEC
    print(f"Loaded: {EDF_FILE.name} | Channel: {channel} | Seizures: {len(seizures)}")

    df = make_windows(signal_uv, fs)

    # Priority 5: Basic baseline signal quality check
    baseline = df[(df.start_s >= baseline_start) & (df.end_s <= baseline_end)]
    if baseline["line_length"].median() > 10000:
        raise ValueError("Baseline window has severe artifact / electrode railing. Recheck baseline.")

    slow_limit = min(0.98, upper_threshold(baseline.slow_ratio))
    line_limit = upper_threshold(baseline.line_length)

    df["seizure_window"] = [
        any(start < row.end_s <= end for start, end in seizures)
        for row in df.itertuples()
    ]

    # Priority 1: Compare Line-Length-only, Slow-Wave-only, and Combined rules
    rule_line = (df.line_length > line_limit).rolling(PERSISTENCE_COUNT, min_periods=PERSISTENCE_COUNT).sum().eq(PERSISTENCE_COUNT)
    rule_slow = (df.slow_ratio > slow_limit).rolling(PERSISTENCE_COUNT, min_periods=PERSISTENCE_COUNT).sum().eq(PERSISTENCE_COUNT)
    rule_and = ((df.line_length > line_limit) & (df.slow_ratio > slow_limit)).rolling(PERSISTENCE_COUNT, min_periods=PERSISTENCE_COUNT).sum().eq(PERSISTENCE_COUNT)

    print("\n" + "=" * 58)
    print("        FEATURE COMPARISON EVALUATION (POST-BASELINE)")
    print("=" * 58)
    for name, series in [("Line Length Only", rule_line),
                         ("Slow Ratio Only", rule_slow),
                         ("Combined (AND Rule)", rule_and)]:
        det, tot, far = evaluate_rule(df, series, seizures, baseline_end, recording_sec)
        sens_pct = (det / tot * 100) if tot > 0 else 0.0
        print(f"{name:<22} | Sensitivity: {det}/{tot} ({sens_pct:>5.1f}%) | FAR/hr: {far:>5.2f}")
    print("=" * 58)
    print(f"Alert persistence: {PERSISTENCE_COUNT} consecutive overlapping windows (spans 7.0 seconds of EEG).")

    # Save visualization for best-performing rule
    df["slow_limit"] = slow_limit
    df["line_limit"] = line_limit
    time_min = (df["start_s"] + WINDOW_SEC / 2) / 60
    fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
    axes[0].plot(time_min, df["slow_ratio"], label="Slow-Wave Ratio", color="teal")
    axes[0].axhline(slow_limit, linestyle="--", color="red", label="Threshold")
    axes[0].set_ylabel("1-8 Hz / 1-30 Hz")
    axes[0].legend(loc="upper right")

    axes[1].plot(time_min, df["line_length"], label="Line Length", color="navy")
    axes[1].axhline(line_limit, linestyle="--", color="red", label="Threshold")
    axes[1].set_ylabel("µV/sec")
    axes[1].set_xlabel("Recording time (minutes)")
    axes[1].legend(loc="upper right")

    for ax in axes:
        for a, b in seizures:
            ax.axvspan(a / 60, b / 60, color="orange", alpha=0.3)
        ax.grid(alpha=0.3)

    plt.suptitle(f"AuraBuds CHB-MIT PoC | {channel}")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "eeg_analysis.png", dpi=150)
    plt.close()


if __name__ == "__main__":
    main()

Loaded: chb01_03.edf | Channel: T7-P7 | Seizures: 1

        FEATURE COMPARISON EVALUATION (POST-BASELINE)
Line Length Only       | Sensitivity: 1/1 (100.0%) | FAR/hr: 11.51
Slow Ratio Only        | Sensitivity: 0/1 (  0.0%) | FAR/hr: 77.44
Combined (AND Rule)    | Sensitivity: 0/1 (  0.0%) | FAR/hr:  2.09
Alert persistence: 3 consecutive overlapping windows (spans 7.0 seconds of EEG).
